# Day 78 -- matplotlib Basics: Data Visualization in Python

This notebook covers the five fundamental matplotlib chart types: **line**, **bar**, **scatter**, **histogram**, and **subplots**.  
Each section includes a Python example, an equivalent C++ comparison using gnuplot/matplotlib-cpp, and an enterprise use-case.

## 0. Setup and Configuration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Enable SVG output for crisp, scalable plots
%config InlineBackend.figure_format='svg'

# Fix Chinese font rendering (if applicable)
try:
    plt.rcParams['font.sans-serif'].insert(0, 'SimHei')
except Exception:
    pass
plt.rcParams['axes.unicode_minus'] = False

print(f'numpy {np.__version__}, matplotlib {plt.matplotlib.__version__}')

---
## 1. Line Plot -- Observing Trends

Line plots are ideal for showing **trends over time**.  
Use `plt.plot()` with `color`, `marker`, `linestyle`, and `linewidth` to customize appearance.

**C++ comparison:** In C++ with gnuplot-iostream, the same chart requires:
```cpp
#include <gnuplot-iostream.h>
#include <vector>
#include <cmath>

int main() {
    Gnuplot gp;
    std::vector<double> x, y;
    for (double v = -6.28; v <= 6.28; v += 0.05) {
        x.push_back(v);
        y.push_back(std::sin(v));
    }
    gp << "set terminal svg\n";
    gp << "set output 'sin.svg'\n";
    gp << "plot '-' with linespoints title 'sin(x)'\n";
    gp.send1d(std::make_tuple(x, y));
}
```
The C++ version requires explicit memory management, manual loop iteration, and external tool piping -- Python handles this in 3 lines.

In [ ]:
# --- 1a. Basic sine curve ---
x = np.linspace(-2 * np.pi, 2 * np.pi, 120)
y = np.sin(x)

plt.figure(figsize=(8, 4), dpi=120)
plt.plot(x, y, linewidth=2, marker='*', color='red', label='sin(x)')
plt.title('Sine Curve')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# --- 1b. Sine + Cosine with annotations ---
y1, y2 = np.sin(x), np.cos(x)

plt.figure(figsize=(8, 4), dpi=120)
plt.plot(x, y1, linewidth=2, marker='*', color='red', label='sin(x)')
plt.plot(x, y2, linewidth=2, marker='^', color='blue', linestyle='--', label='cos(x)')

plt.annotate('sin(x)', xytext=(0.5, -0.75), xy=(0, -0.25), fontsize=12,
             arrowprops={'arrowstyle': '->', 'color': 'darkgreen',
                         'connectionstyle': 'angle3, angleA=90, angleB=0'})
plt.annotate('cos(x)', xytext=(-3, 0.75), xy=(-1.25, 0.5), fontsize=12,
             arrowprops={'arrowstyle': '->', 'color': 'darkgreen',
                         'connectionstyle': 'arc3, rad=0.35'})

plt.title('Sine and Cosine Curves')
plt.xlabel('x (radians)')
plt.ylabel('Amplitude')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Enterprise Example: Stock Price Trend

Financial dashboards plot daily closing prices as line charts to identify trends.
The C++ equivalent would need a CSV parser (e.g., `rapidcsv`) + gnuplot pipe -- Python's `pandas.plot()` wraps this in a single call.

In [ ]:
# Simulated 90-day stock price with a random walk
np.random.seed(42)
days = np.arange(1, 91)
price = 100 + np.cumsum(np.random.randn(90) * 1.2)

plt.figure(figsize=(10, 4), dpi=120)
plt.plot(days, price, color='#1f77b4', linewidth=1.5)
plt.fill_between(days, price, alpha=0.15, color='#1f77b4')
plt.title('Simulated Stock Closing Price (90 Days)')
plt.xlabel('Trading Day')
plt.ylabel('Price ($)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 2. Bar Plot -- Comparing Categories

Bar plots are the best choice for **comparing discrete categories**.  
Use `plt.bar()` for vertical bars and `plt.barh()` for horizontal bars.

**C++ comparison:** With matplotlib-cpp:
```cpp
#include <matplotlibcpp.h>
namespace plt = matplotlibcpp;

int main() {
    std::vector<double> x{0,1,2,3};
    std::vector<double> y1{35, 48, 22, 40};
    std::vector<double> y2{28, 35, 45, 30};
    // No built-in grouped bar -- must manually offset each group
    std::vector<double> x1, x2;
    for (auto v : x) { x1.push_back(v - 0.1); x2.push_back(v + 0.1); }
    plt::bar(x1, y1, 0.2);
    plt::bar(x2, y2, 0.2);
    plt::show();
}
```
Python's `plt.bar()` returns a container object that simplifies label placement and legend binding.

In [ ]:
# --- 2a. Grouped bar chart ---
x = np.arange(4)
np.random.seed(7)
y1 = np.random.randint(20, 50, 4)
y2 = np.random.randint(10, 60, 4)

plt.figure(figsize=(7, 4), dpi=120)
plt.bar(x - 0.1, y1, width=0.2, label='Team A', color='steelblue')
plt.bar(x + 0.1, y2, width=0.2, label='Team B', color='coral')
plt.xticks(x, labels=['Q1', 'Q2', 'Q3', 'Q4'])
plt.ylabel('Revenue (k$)')
plt.title('Quarterly Sales by Team')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- 2b. Stacked bar chart ---
labels = ['Q1', 'Q2', 'Q3', 'Q4']

plt.figure(figsize=(7, 4), dpi=120)
plt.bar(labels, y1, width=0.4, label='Team A', color='steelblue')
plt.bar(labels, y2, width=0.4, bottom=y1, label='Team B', color='coral')
plt.ylabel('Revenue (k$)')
plt.title('Quarterly Sales -- Stacked')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

### Enterprise Example: Monthly Active Users by Platform

Product managers compare web vs. mobile vs. desktop MAU using grouped bars.

In [ ]:
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun']
web    = np.array([120, 135, 128, 142, 155, 160])
mobile = np.array([200, 215, 230, 245, 260, 280])
desktop = np.array([45, 42, 40, 38, 35, 33])

x_pos = np.arange(len(months))
w = 0.25

plt.figure(figsize=(9, 5), dpi=120)
plt.bar(x_pos - w, web,     w, label='Web',     color='#4e79a7')
plt.bar(x_pos,     mobile,  w, label='Mobile',  color='#f28e2b')
plt.bar(x_pos + w, desktop, w, label='Desktop', color='#e15759')
plt.xticks(x_pos, months)
plt.ylabel('MAU (thousands)')
plt.title('Monthly Active Users by Platform')
plt.legend()
plt.tight_layout()
plt.show()

---
## 3. Scatter Plot -- Finding Relationships

Scatter plots reveal **correlations between two variables**.  
Use `plt.scatter()` with `s` (size), `c` (color), and `alpha` for richer encoding.

**C++ comparison:** With gnuplot-iostream, scatter requires `with points` and manual color arrays:
```cpp
gp << "plot '-' with points pt 7 ps 1.5 title 'data'\n";
gp.send1d(std::make_tuple(x, y));
```
Python's `plt.scatter()` accepts NumPy arrays directly and maps colors via a colormap in one call.

In [ ]:
# --- 3a. Income vs. online spending ---
income   = np.array([5550, 7500, 10500, 15000, 20000, 25000, 30000, 40000])
spending = np.array([800, 1800, 1250, 2000, 1800, 2100, 2500, 3500])

plt.figure(figsize=(7, 4), dpi=120)
plt.scatter(income, spending, s=80, color='darkcyan', edgecolors='black', linewidth=0.5)
plt.xlabel('Monthly Income ($)')
plt.ylabel('Online Spending ($)')
plt.title('Income vs. Online Spending')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# --- 3b. Bubble chart (3 variables) ---
np.random.seed(0)
n = 50
x = np.random.randn(n) * 10 + 50   # temperature
y = np.random.randn(n) * 15 + 60   # humidity
sizes = np.random.randint(20, 200, n)  # energy consumption
colors = np.random.rand(n)

plt.figure(figsize=(7, 5), dpi=120)
plt.scatter(x, y, s=sizes, c=colors, alpha=0.6, cmap='viridis', edgecolors='gray', linewidth=0.5)
plt.colorbar(label='Color Index')
plt.xlabel('Temperature (F)')
plt.ylabel('Humidity (%)')
plt.title('Energy Consumption Bubble Chart')
plt.tight_layout()
plt.show()

### Enterprise Example: Server Latency vs. Request Rate
SRE teams scatter-plot p99 latency against QPS to spot saturation points.

In [ ]:
np.random.seed(10)
qps = np.random.uniform(100, 5000, 80)
# latency grows super-linearly after ~3000 QPS
latency = 20 + 0.005 * qps + 0.000002 * qps**2 + np.random.randn(80) * 8

plt.figure(figsize=(8, 5), dpi=120)
sc = plt.scatter(qps, latency, c=latency, cmap='RdYlGn_r', s=40, alpha=0.8, edgecolors='k', linewidth=0.3)
plt.colorbar(sc, label='p99 Latency (ms)')
plt.axhline(y=100, color='red', linestyle='--', linewidth=1, label='SLA = 100 ms')
plt.xlabel('Requests per Second (QPS)')
plt.ylabel('p99 Latency (ms)')
plt.title('API Latency vs. Request Rate')
plt.legend()
plt.tight_layout()
plt.show()

---
## 4. Histogram -- Understanding Distributions

Histograms show **how data is distributed** across value ranges.  
Use `plt.hist()` with `bins`, `density`, and `cumulative` parameters.

**C++ comparison:** In C++ you would manually count bin frequencies:
```cpp
std::map<int,int> bins;
for (auto h : heights) bins[h / 5 * 5]++;
// then pipe to gnuplot as a bar chart
```
Python's `plt.hist()` computes bins, counts, and renders in a single call.

In [ ]:
# --- 4a. Height distribution (100 male students) ---
heights = np.array([
    170, 163, 174, 164, 159, 168, 165, 171, 171, 167,
    165, 161, 175, 170, 174, 170, 174, 170, 173, 173,
    167, 169, 173, 153, 165, 169, 158, 166, 164, 173,
    162, 171, 173, 171, 165, 152, 163, 170, 171, 163,
    165, 166, 155, 155, 171, 161, 167, 172, 164, 155,
    168, 171, 173, 169, 165, 162, 168, 177, 174, 178,
    161, 180, 155, 155, 166, 175, 159, 169, 165, 174,
    175, 160, 152, 168, 164, 175, 168, 183, 166, 166,
    182, 174, 167, 168, 176, 170, 169, 173, 177, 168,
    172, 159, 173, 185, 161, 170, 170, 184, 171, 172
])

plt.figure(figsize=(7, 4), dpi=120)
plt.hist(heights, bins=np.arange(145, 196, 5), color='darkcyan', edgecolor='white')
plt.xlabel('Height (cm)')
plt.ylabel('Count')
plt.title('Height Distribution of 100 Male Students')
plt.tight_layout()
plt.show()

In [ ]:
# --- 4b. Cumulative density histogram ---
plt.figure(figsize=(7, 4), dpi=120)
plt.hist(heights, bins=np.arange(145, 196, 5), color='darkcyan', edgecolor='white',
         density=True, cumulative=True)
plt.xlabel('Height (cm)')
plt.ylabel('Cumulative Probability')
plt.title('Cumulative Height Distribution')
plt.tight_layout()
plt.show()

### Enterprise Example: Response Time Distribution
Operations teams histogram response times to verify SLA compliance.

In [ ]:
np.random.seed(3)
# Log-normal response times (typical for web services)
response_ms = np.random.lognormal(mean=3.5, sigma=0.6, size=2000)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), dpi=120)

# Left: raw histogram
axes[0].hist(response_ms, bins=50, color='#4e79a7', edgecolor='white', alpha=0.85)
axes[0].axvline(x=np.percentile(response_ms, 99), color='red', linestyle='--', label='p99')
axes[0].set_xlabel('Response Time (ms)')
axes[0].set_ylabel('Count')
axes[0].set_title('Response Time Distribution')
axes[0].legend()

# Right: cumulative
axes[1].hist(response_ms, bins=50, color='#f28e2b', edgecolor='white', alpha=0.85,
             density=True, cumulative=True)
axes[1].axhline(y=0.99, color='red', linestyle='--', label='99th percentile')
axes[1].set_xlabel('Response Time (ms)')
axes[1].set_ylabel('Cumulative Probability')
axes[1].set_title('Cumulative Response Time')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 5. Subplots -- Combining Multiple Charts

Subplots let you place **multiple charts in one figure** for comparison.  
Use `plt.subplot(rows, cols, index)` or `fig, axes = plt.subplots()` for full control.

**C++ comparison:** With matplotlib-cpp, subplots require:
```cpp
plt::subplot(2, 1, 1);
plt::plot(x, y1);
plt::subplot(2, 1, 2);
plt::plot(x, y2);
plt::show();
```
Similar API but fewer layout options (no `constrained_layout`, no `GridSpec`).

In [ ]:
# --- 5a. Basic subplot with plt.subplot() ---
x = np.linspace(-2 * np.pi, 2 * np.pi, 120)
y1, y2 = np.sin(x), np.cos(x)

plt.figure(figsize=(8, 5), dpi=120)

plt.subplot(2, 1, 1)
plt.plot(x, y1, linewidth=2, marker='*', color='red', markersize=4)
plt.title('sin(x)')
plt.grid(True, alpha=0.3)

plt.subplot(2, 1, 2)
plt.plot(x, y2, linewidth=2, marker='^', color='blue', markersize=4)
plt.title('cos(x)')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# --- 5b. Side-by-side subplots ---
plt.figure(figsize=(10, 4), dpi=120)

plt.subplot(1, 2, 1)
plt.plot(x, y1, linewidth=2, marker='*', color='red', markersize=3)
plt.title('sin(x)')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(x, y2, linewidth=2, marker='^', color='blue', markersize=3)
plt.title('cos(x)')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# --- 5c. Object-oriented subplots with fig, axes ---
fig, axes = plt.subplots(2, 2, figsize=(9, 7), dpi=120)

# Top-left: line
axes[0, 0].plot(x, y1, color='crimson', linewidth=1.5)
axes[0, 0].set_title('Line: sin(x)')
axes[0, 0].grid(True, alpha=0.3)

# Top-right: bar
categories = ['A', 'B', 'C', 'D']
values = [25, 40, 30, 55]
axes[0, 1].bar(categories, values, color=['#4e79a7', '#f28e2b', '#e15759', '#76b7b2'])
axes[0, 1].set_title('Bar: Category Comparison')

# Bottom-left: scatter
np.random.seed(42)
axes[1, 0].scatter(np.random.randn(60), np.random.randn(60),
                   c=np.random.rand(60), s=50, alpha=0.7, cmap='plasma')
axes[1, 0].set_title('Scatter: Random Points')
axes[1, 0].grid(True, alpha=0.3)

# Bottom-right: histogram
axes[1, 1].hist(np.random.randn(500), bins=25, color='teal', edgecolor='white')
axes[1, 1].set_title('Histogram: Normal Distribution')

fig.suptitle('Four Chart Types in One Figure', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Enterprise Example: Executive Dashboard

A single figure combining KPI line trend, revenue bar comparison, user scatter, and latency histogram -- the kind of dashboard product teams present to leadership.

In [ ]:
np.random.seed(1)
fig, axes = plt.subplots(2, 2, figsize=(11, 8), dpi=120)

# KPI trend (line)
weeks = np.arange(1, 13)
kpi = 85 + np.cumsum(np.random.randn(12) * 2)
axes[0, 0].plot(weeks, kpi, marker='o', color='#2ca02c', linewidth=2)
axes[0, 0].fill_between(weeks, kpi, alpha=0.15, color='#2ca02c')
axes[0, 0].axhline(y=90, color='gray', linestyle=':', label='Target')
axes[0, 0].set_title('KPI Score Trend')
axes[0, 0].set_xlabel('Week')
axes[0, 0].set_ylabel('KPI')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Revenue by region (bar)
regions = ['NA', 'EU', 'APAC', 'LATAM']
revenue = [420, 310, 280, 95]
bars = axes[0, 1].bar(regions, revenue, color=['#4e79a7', '#f28e2b', '#e15759', '#76b7b2'])
axes[0, 1].bar_label(bars, fmt='$%dk')
axes[0, 1].set_title('Revenue by Region')
axes[0, 1].set_ylabel('Revenue (k$)')

# User growth vs. churn (scatter)
growth = np.random.uniform(2, 15, 20)
churn = 20 - growth * 0.8 + np.random.randn(20) * 2
axes[1, 0].scatter(growth, churn, c=churn, cmap='RdYlGn', s=70, edgecolors='k', linewidth=0.5)
axes[1, 0].set_title('Growth vs. Churn Rate')
axes[1, 0].set_xlabel('Growth %')
axes[1, 0].set_ylabel('Churn %')
axes[1, 0].grid(True, alpha=0.3)

# Latency distribution (histogram)
latencies = np.random.lognormal(3.2, 0.5, 3000)
axes[1, 1].hist(latencies, bins=40, color='mediumpurple', edgecolor='white')
axes[1, 1].axvline(x=np.percentile(latencies, 95), color='red', linestyle='--', label='p95')
axes[1, 1].set_title('API Latency Distribution')
axes[1, 1].set_xlabel('Latency (ms)')
axes[1, 1].set_ylabel('Count')
axes[1, 1].legend()

fig.suptitle('Executive Dashboard -- Q2 Summary', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## 6. Saving Charts

Always call `plt.savefig()` **before** `plt.show()` -- `show()` releases the figure, so any save after it produces a blank image.

**C++ comparison:** gnuplot uses `set terminal png; set output 'file.png'` at the top of the script; matplotlib-cpp wraps this as `plt::save("file.png")`.

In [ ]:
plt.figure(figsize=(6, 3), dpi=120)
plt.plot([1, 2, 3, 4], [10, 20, 25, 30], marker='o', color='steelblue')
plt.title('Savefig Demo')

# Save BEFORE show
plt.savefig('demo_chart.png', bbox_inches='tight', dpi=150)
plt.show()
print('Chart saved to demo_chart.png')

---
## Summary

| Chart Type | Function | Best For | C++ Equivalent |
|---|---|---|---|
| Line | `plt.plot()` | Trends over time | gnuplot `with linespoints` |
| Bar | `plt.bar()` / `plt.barh()` | Category comparison | matplotlib-cpp `plt::bar()` |
| Scatter | `plt.scatter()` | Variable correlation | gnuplot `with points` |
| Histogram | `plt.hist()` | Data distribution | Manual bin counting + bar |
| Subplots | `plt.subplot()` / `plt.subplots()` | Multi-chart layout | `plt::subplot()` |

**Key takeaway:** Python + matplotlib achieves in 3-5 lines what C++ requires 15-30 lines for, with richer defaults and seamless NumPy/Pandas integration.